# 06 - Structured Output 與風險比較

當 agent 從文件萃取資訊時，有兩種做法：

| 做法 | 方式 | 問題 |
|------|------|------|
| **無結構** | 直接問 LLM「幫我列出決策」，得到純文字 | 格式不定、欄位可能缺失、難以程式化處理 |
| **有結構** | `with_structured_output(PydanticModel)` | LLM 被迫輸出符合 schema 的 JSON，可直接使用 |

`with_structured_output()` 是 LangChain 的方法，用 Pydantic `BaseModel` 定義 schema。
它在底層使用 function calling 強制 LLM 輸出特定結構，**不需要額外安裝套件**（Pydantic 已是 LangChain 的依賴）。

In [1]:
import os
import sys
from pathlib import Path

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / 'examples'
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from utils.tools import FileTools

load_dotenv(PROJECT_ROOT / '.env')

llm = ChatOpenAI(
    base_url=os.environ['VLLM_BASE_URL'],
    api_key=os.environ['VLLM_API_KEY'],
    model=os.environ['VLLM_MODEL'],
    temperature=0,
)
_crm = FileTools(PROJECT_ROOT / 'data' / 'crm_notes')
print(f'Project root: {PROJECT_ROOT}')
print(f'LLM: {os.environ["VLLM_MODEL"]}')

Project root: /home/mistin/research/agentic-rag-lab
LLM: gemma-4-31B-it


## 1 · 定義 Pydantic Schema

三個 schema 對應會議紀錄中的三種結構化資訊：

- `DecisionRecord` — 決策清單（ID、內容、決策人、日期）
- `RiskRecord` — 風險清單（ID、描述、嚴重度、機率）
- `ActionRecord` — 行動項目（負責人、內容、截止日期）

In [2]:
from pydantic import BaseModel, Field


class Decision(BaseModel):
    id: str = Field(description='決策編號，如 D-001')
    content: str = Field(description='決策內容摘要')
    owner: str = Field(description='決策人姓名')
    date: str = Field(description='決策日期，格式 YYYY-MM-DD')


class DecisionRecord(BaseModel):
    customer: str = Field(description='客戶名稱')
    meeting_date: str = Field(description='會議日期')
    decisions: list[Decision] = Field(description='本次會議的所有決策')


class Risk(BaseModel):
    id: str = Field(description='風險編號，如 R-001')
    description: str = Field(description='風險描述')
    severity: str = Field(description='嚴重度：高/中/低')
    probability: str = Field(description='發生機率：高/中/低')


class RiskRecord(BaseModel):
    customer: str
    meeting_date: str
    risks: list[Risk]


class ActionItem(BaseModel):
    owner: str = Field(description='負責人')
    content: str = Field(description='行動項目描述')
    due_date: str = Field(description='截止日期，格式 YYYY-MM-DD，若無則填 unknown')

In [3]:
class ActionRecord(BaseModel):
    customer: str
    meeting_date: str
    actions: list[ActionItem]

## 2 · with_structured_output() 示範

`llm.with_structured_output(Schema)` 回傳一個新的 runnable，
呼叫 `.invoke()` 後直接得到 Pydantic 物件，而不是字串。

In [4]:
decision_extractor = llm.with_structured_output(DecisionRecord)

doc_001 = _crm.read_file('meeting_001_晨星科技_2026-05-08.md')
result_001: DecisionRecord = decision_extractor.invoke(
    f'請從以下會議紀錄萃取所有決策：\n\n{doc_001}'
)

print('=== 決策萃取（meeting_001） ===')
print(f'客戶: {result_001.customer}')
print(f'會議日期: {result_001.meeting_date}')
print(f'決策數: {len(result_001.decisions)}')
print()
for d in result_001.decisions:
    print(f'  {d.id} | {d.content:<36} | {d.owner:<14} | {d.date}')

=== 決策萃取（meeting_001） ===
客戶: 晨星科技股份有限公司
會議日期: 2026-05-08
決策數: 4

  D-001 | 採用方案 A（公有雲部署，AWS 台灣區），以 SaaS 模式導入    | 李明哲、王雅婷        | 2026-05-08
  D-002 | 第一階段範疇鎖定財務模組，人資與供應鏈模組列為第二階段          | 李明哲            | 2026-05-08
  D-003 | 目標上線日期定為 2026-07-01（財務模組 Go-Live）    | 李明哲            | 2026-05-08
  D-004 | 技術規格書由我方（陳建宏）撰寫，客戶方（張志偉）負責確認         | 王雅婷            | 2026-05-08


In [5]:
risk_extractor = llm.with_structured_output(RiskRecord)

doc_003 = _crm.read_file('meeting_003_晨星科技_2026-05-22.md')
result_003: RiskRecord = risk_extractor.invoke(
    f'請從以下會議紀錄萃取所有風險項目：\n\n{doc_003}'
)

print('=== 風險萃取（meeting_003） ===')
print(f'客戶: {result_003.customer}')
print(f'會議日期: {result_003.meeting_date}')
print()
for r in result_003.risks:
    print(f'  {r.id} | {r.description:<32} | 嚴重度={r.severity} | 機率={r.probability}')

=== 風險萃取（meeting_003） ===
客戶: 晨星科技股份有限公司
會議日期: 2026-05-22

  R-001 | 歷史資料遷移失真                         | 嚴重度=高 | 機率=中
  R-002 | 銀行 API 整合                        | 嚴重度=高 | 機率=高
  R-003 | 部署方案變更導致工期重算，原定 2026-07-01 上線風險極高（且後續決策要求提前至 06-15，壓力進一步增加） | 嚴重度=高 | 機率=高
  R-004 | 客戶 UAT 學習曲線                      | 嚴重度=低 | 機率=高
  R-005 | 私有雲硬體採購週期（6–8 週）可能成為關鍵瓶頸         | 嚴重度=高 | 機率=高
  R-006 | PM 人力不足（王雅婷同時負責鴻圖零售專案，時程壓縮導致負荷過重） | 嚴重度=中 | 機率=高
  R-007 (隱含) | 資料遷移驗證次數從 3 次縮減為 1 次，若出現資料異常將無充足時間修復 | 嚴重度=高 | 機率=未標記 (技術負責人警告)
  R-008 (隱含) | 技術負責人（陳建宏）人力衝突，本案 Go-Live 後的支援期可能與新星金融科技專案重疊 | 嚴重度=未標記 | 機率=未標記


## 3 · 無結構化輸出的風險

不使用 `with_structured_output()`，直接問 LLM 同一個問題。
比較兩種輸出的**可程式化程度**。

In [6]:
# 無結構版本：直接呼叫 LLM，得到純文字
plain_llm = ChatOpenAI(
    base_url=os.environ['VLLM_BASE_URL'],
    api_key=os.environ['VLLM_API_KEY'],
    model=os.environ['VLLM_MODEL'],
    temperature=0,
)
plain_result = plain_llm.invoke(f'請從以下會議紀錄萃取所有決策：\n\n{doc_001}')
plain_text = plain_result.content

print('=== 無結構輸出（純文字） ===')
print(plain_text[:400])

print()
print('>>> 嘗試程式化存取 decisions[0].id ...')
try:
    _ = plain_text.decisions[0].id  # type: ignore
except AttributeError as e:
    print(f'AttributeError: {e}')

=== 無結構輸出（純文字） ===
本次會議的所有決策萃取如下：

**【決策清單】**

1. **部署方案**：採用**方案 A**（公有雲部署，AWS 台灣區），以 **SaaS 模式**導入。
2. **實施範疇**：採取分階段上線，**第一階段鎖定「財務模組」**；人資與供應鏈模組則移至第二階段。
3. **上線時程**：目標正式上線（Go-Live）日期定為 **2026-07-01**（針對財務模組）。
4. **文件分工**：**技術規格書**由我方（陳建宏）負責撰寫，並由客戶方（張志偉）負責確認簽核。

>>> 嘗試程式化存取 decisions[0].id ...
AttributeError: 'str' object has no attribute 'decisions'


### 三種常見風險

| 風險 | 無結構 | 有結構（Pydantic） |
|------|--------|--------------------|
| **欄位缺失** | LLM 可能跳過「不重要」的欄位 | 欄位為必填，缺少時會重試 |
| **格式不一** | 每次輸出可能用不同的 markdown 格式 | JSON schema 強制固定格式 |
| **類型錯誤** | 日期可能輸出「五月八日」而非 `2026-05-08` | `Field(description=...)` 提供格式說明 |

當萃取結果需要被其他程式碼使用（如比較兩份文件的決策、寫入資料庫），
無結構輸出需要額外的 parsing 步驟，且容易出錯。

## 4 · 用 05 的評估邏輯量化品質差異

針對一個需要「萃取後再回答」的問題，比較：
- **基線**：直接問 LLM（無結構）
- **有結構**：先 `with_structured_output()` 萃取，再格式化成清晰回答

評估用 05 的 `llm_judge()` 邏輯。

In [7]:
import json as _json
from openai import OpenAI as _OpenAI

_openai_client = _OpenAI(
    base_url=os.environ['VLLM_BASE_URL'],
    api_key=os.environ['VLLM_API_KEY'],
)

QUESTION = 'meeting_001 中有哪些決策？請列出決策編號、內容與決策人。'
KEYWORDS = ['D-001', 'D-002', 'D-003', 'D-004', '公有雲', '財務模組', '陳建宏']

print(f'問題：{QUESTION}')
print(f'關鍵字：{", ".join(KEYWORDS)}')


def quick_judge(question: str, keywords: list[str], answer: str) -> tuple[int, str]:
    kw_str = ', '.join(f'「{k}」' for k in keywords)
    prompt = (
        f'問題：{question}\n\n'
        f'正確答案應包含：{kw_str}\n\n'
        f'Agent 回答：\n{answer}\n\n'
        '請只輸出 JSON：{"score": <0|1|2>, "reason": "<一句話>"}'
    )
    resp = _openai_client.chat.completions.create(
        model=os.environ['VLLM_MODEL'],
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    import re
    raw = resp.choices[0].message.content or ''
    m = re.search(r'\{[^}]+\}', raw)
    if m:
        data = _json.loads(m.group())
        return int(data.get('score', 0)), data.get('reason', '')
    return 0, raw[:80]


# 無結構版
plain_answer = plain_llm.invoke(f'請回答以下問題，依據這份文件：\n{doc_001}\n\n問題：{QUESTION}').content
score_plain, reason_plain = quick_judge(QUESTION, KEYWORDS, plain_answer)

print()
print('── 無結構輸出 ──────────────────────────────────────')
print(f'輸出長度: {len(plain_answer)} chars')
print(f'judge_score: {score_plain}  |  reason: {reason_plain}')
print('可程式化存取 D-001 內容: ✗ 需要手動 parsing')

# 有結構版：從 result_001（已在 §2 萃取）格式化輸出
structured_answer = '\n'.join(
    f'{d.id}：{d.content}（決策人：{d.owner}，{d.date}）'
    for d in result_001.decisions
)
score_struct, reason_struct = quick_judge(QUESTION, KEYWORDS, structured_answer)

print()
print('── 有結構輸出 ──────────────────────────────────────')
print(f'輸出長度: {len(structured_answer)} chars（從 Pydantic 物件格式化）')
print(f'judge_score: {score_struct}  |  reason: {reason_struct}')
d0 = result_001.decisions[0]
print(f'可程式化存取 D-001 內容: ✓ result_001.decisions[0].content = {d0.content!r}')

問題：meeting_001 中有哪些決策？請列出決策編號、內容與決策人。
關鍵字：D-001, D-002, D-003, D-004, 公有雲, 財務模組, 陳建宏

── 無結構輸出 ──────────────────────────────────────
輸出長度: 279 chars
judge_score: 2  |  reason: Agent 回答完整包含了所有要求的決策編號、關鍵內容（公有雲、財務模組）以及決策人（陳建宏）。
可程式化存取 D-001 內容: ✗ 需要手動 parsing

── 有結構輸出 ──────────────────────────────────────
輸出長度: 232 chars（從 Pydantic 物件格式化）
judge_score: 2  |  reason: 回答完整包含所有要求的決策編號、關鍵內容（公有雲、財務模組）及決策人（陳建宏）。
可程式化存取 D-001 內容: ✓ result_001.decisions[0].content = '採用方案 A（公有雲部署，AWS 台灣區），以 SaaS 模式導入'


## 小結

`with_structured_output()` 在以下情境下最有價值：

1. **萃取後需要程式化處理**：如比較兩份文件的決策（10 節）
2. **下游系統需要固定格式**：如寫入 JSONL、送進資料庫
3. **欄位完整性有要求**：Pydantic 的 `Field(description=...)` 提供格式說明給 LLM

純問答場景（judge_score 相近）中兩者品質差異不大，但**結構化輸出讓後續處理更可靠**。

---
**下一個筆記本（07）**：LangGraph Checkpointing，讓 agent 能記住多輪對話的脈絡。